<img src="https://userweb.fct.unl.pt/~jmc.xavier/MAI/iLogos/logo_novafct.png" width="200">

# Departamento de Engenharia Mecânica e Industrial
## Mecânica Aplicada I 

## ANÁLISE DE TRELIÇAS


### Problema 8

Na Figura está representada uma treliça constituída por 17 barras. No ponto A existe um apoio fixo, no ponto E existe um apoio móvel. Considere o carregamento indicado, e despreze o peso das barras. 

[a)] Trata-se de uma treliça simples? Justifique.
[b)] Calcule as componentes da reação em cada um dos apoios.
[c)] Justifique a seguinte afirmação: ``As barras CD e DE são barras de esforço nulo para o carregamento indicado.''. Qual a finalidade de uma estrutura ter barras com esforço nulo?.
[d)] Determine o esforço nas barras JE, JF e GF, identificando se estão à tração ou compressão.  

<img src="https://userweb.fct.unl.pt/~jmc.xavier/MAI/Notebooks/Ch07/P8/MAI_Ch07_P8.png" width="620">


**Estatia global**

Estatia global da treliça ($\alpha_g$) é calculada de acordo com:

\begin{equation*}
\alpha_g = \alpha_e + \alpha_i
\end{equation*}

em que $\alpha_e$ e $\alpha_i$ são a estatia exterior e estatia interior da treliça calculadas, respectivamente, por:

\begin{equation*}
\alpha_e = 3 × \textrm{( nº de apoios − 1)} − \textrm{nº de libertações exteriores}
\end{equation*}

- Apoios: O número de apoios refere-se ao total de apoios que a treliça tem
- Libertações Exteriores: Este termo refere-se ao número de graus de liberdade que não estão restringidos pelos apoios

\begin{equation*}
\alpha_i = 3 × \textrm{nº de malhas fechadas} − \textrm{nº de ligações interiores}
\end{equation*}

- Malha fechada numa treliça refere-se a uma região da estrutura que é completamente delimitada por barras, formando um circuito fechado. Malhas fechadas considera que cada malha adiciona três graus de liberdade internos.
- Ligações interiores referem-se às barras internas da treliça que conectam os nós entre si.

In [1]:
def estatiaexterior(napoios, nlibertacoes):
    return 3*(napoios-1) - nlibertacoes

def estatiainterior(nmalhas,nligainter):
    return  3*nmalhas - nligainter

npoios_, nlibext_ = 2, 3
alphe = estatiaexterior(npoios_,nlibext_)
print(f'estatia exterior = 3({npoios_} -1) - {nlibext_} =  {alphe}')
nmalh_, nlibint_ = 8, 1*2 + 2*4 + 3*2 + 4*2 
alphi = estatiainterior(nmalh_,nlibint_)
print(f'estatia interior = 3({nmalh_}) - {nlibint_} = {alphi}')
estatiaglobal = alphe + alphi
print((f'estatia global = {estatiaglobal}'))


estatia exterior = 3(2 -1) - 3 =  0
estatia interior = 3(8) - 24 = 0
estatia global = 0


In [30]:
import numpy as np
import sympy as sy
from sympy.solvers import solve
from typing import NamedTuple

# unit: m, KN

# Dados
# unidades SI: MPa, m, s, kg/m^3
class dados(NamedTuple):
    H : float 
    L : float
    F : float

var = dados(1.5, 2.0, 6.0)
print(f'dados = {var}')

rax, ray, rey = sy.symbols('rax ray rey')

print('-------------------')
print('Sistema de Equações')
print('-------------------')

SFX = rax
print(f'sum Fx : {SFX} = 0')
SFY = ray + rey - 2*var.F
print(f'sum Fy : {SFY} = 0')
SMA = -var.F * (2 * var.H) - var.F * (3 * var.H) + rey*(4 * var.H)
print(f'sum MD : {SMA} = 0')
sol = solve({SFX, SFY, SMA}, {rax, ray, rey})

print('\n------')
print('Reações')
print('-------')

RAx = sol[rax]
print(f'RAx = {RAx} kN')
RAy = sol[ray]
print(f'RAy = {sy.N(RAy):.1f} kN')
REy = sol[rey]
print(f'REy = {sy.N(REy):.1f} kN')


dados = dados(H=1.5, L=2.0, F=6.0)
-------------------
Sistema de Equações
-------------------
sum Fx : rax = 0
sum Fy : ray + rey - 12.0 = 0
sum MD : 6.0*rey - 45.0 = 0

------
Reações
-------
RAx = 0.0 kN
RAy = 4.5 kN
REy = 7.5 kN


In [31]:
print('No E')

aJEF = np.arctan(var.L/var.H)
print(f'aJEF = {aJEF:.3f} radianos = {np.rad2deg(aJEF):.1f} graus')

fje, ffe, t = sy.symbols('fje ffe t')

SFX = -fje*sy.cos(t) - ffe
print('sum Fx :', SFX)
SFY = rey + fje*sy.sin(t)
print('sum Fy :', SFY)
sol = solve({SFX,SFY}, {fje, ffe})

FJE = sol[fje]
NFJE = FJE.subs({(t,aJEF),(rey,REy)})
print(f'FJE    = {FJE} = {NFJE:.2f} kN (C)')
FFE = sol[ffe]
NFFE = FFE.subs({(t,aJEF),(rey,REy)})
print(f'FFE    = {FFE} = {NFFE:.2f} kN (T) ')

No E
aJEF = 0.927 radianos = 53.1 graus
sum Fx : -ffe - fje*cos(t)
sum Fy : fje*sin(t) + rey
FJE    = -rey/sin(t) = -9.38 kN (C)
FFE    = rey*cos(t)/sin(t) = 5.62 kN (T) 


In [32]:
print('No F')

fjf, fgf, t = sy.symbols('fjf fgf t')

SFX = NFFE - fgf
print('sum Fx :', SFX)
SFY = -var.F + fjf
print('sum Fy :', SFY)
sol = solve({SFX,SFY}, {fjf, fgf})

FJF = sol[fjf]
print(f'FJE    = {FJF:.2f} kN (T)')
FGF = sol[fgf]
print(f'FGF    = {FGF:.2f} kN (T) ')

No F
sum Fx : 5.625 - fgf
sum Fy : fjf - 6.0
FJE    = 6.00 kN (T)
FGF    = 5.62 kN (T) 


## FTool

- Implimentação:

<img src="https://userweb.fct.unl.pt/~jmc.xavier/MAI/Notebooks/Ch07/P8/FTool00_MAI_Ch07_P8.png" width="420">

- Diagrama dos esforços axiais:

<img src="https://userweb.fct.unl.pt/~jmc.xavier/MAI/Notebooks/Ch07/P8/FTool01_MAI_Ch07_P8.png" width="420">

- Deformada da estrutura:

<img src="https://userweb.fct.unl.pt/~jmc.xavier/MAI/Notebooks/Ch07/P8/FTool02_MAI_Ch07_P8.png" width="420">



---

Copyright (c) DEMI - FCT NOVA

Interactive computing by <a href="https://jupyter.org/" target="_blank"> <span
style="color:#333399"> Jupyter Notebook </span> </a> &nbsp;|&nbsp;Coded by <a href = "mailto: jmc.xavier@fct.unl.pt">José Xavier</a>

Licensed under  <a href="http://creativecommons.org/licenses/by-sa/4.0/"
target="_blank"> <span style="color:#333399;font-size: 20px"> CC BY-SA 4.0  </span></a>